# Feature Attribution using Raking - v1.0

In [1]:
import sys
import time
import numpy as np
import pandas as pd
import zipfile as zf
import sklearn.ensemble
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from scipy.sparse.linalg import eigs
import xgboost as xgb

In [2]:
!{sys.executable} -m pip install catboost
from catboost import CatBoostClassifier, Pool

# Data preprocessing methods

In [3]:
# return one df without nan's on num_cols
# numerical features: fill nan's with median/mean/mode
def pre_proc_fillna_num_fts(df,num_cols,num_type='mean'):
    df_train= df.copy()

    if(num_type=='median'):
        for col in num_cols:
            ft_median= df_train[col].median()
            df_train[col]= df_train[col].fillna(ft_median)
    elif(num_type=='mode'):
        for col in num_cols:
            ft_mode= df_train[col].value_counts().index[0]
            df_train[col]= df_train[col].fillna(ft_mode)
    else:
        for col in num_cols:
            ft_mean= df_train[col].mean()
            df_train[col]= df_train[col].fillna(ft_mean)

    return df_train

In [4]:
# return one df without nan's on cat_cols
# categorical features: fill nan's with mode/mean/median
def pre_proc_fillna_cat_fts(df,cat_cols,cat_type='mode'):
    df_train= df.copy()
    
    if(cat_type!='mode' and type(df_train[cat_cols[0]].value_counts().index[0])!=type('str')):
        if(cat_type=='mean'):
            for col in cat_cols:
                ft_mean= df_train[col].mean()
                df_train[col]= df_train[col].fillna(ft_mean)
        elif(cat_type=='median'):
            for col in cat_cols:
                ft_median= df_train[col].median()
                df_train[col]= df_train[col].fillna(ft_median)
    else:
        for col in cat_cols:
            ft_mode= df_train[col].value_counts().index[0]
            df_train[col]= df_train[col].fillna(ft_mode)

    return df_train

In [5]:
# n_cols refers only to df's columns with numerical values
def normalize_selected_cols(df, n_cols):
    result= df.copy()
    
    for col in n_cols:
        max_value= df[col].max()
        min_value= df[col].min()
        result[col]= (df[col]- min_value)/ (max_value - min_value)
        
    return result

# Data loading and preprocessing

In [6]:
!kaggle competitions download -c titanic

  0%|                                               | 0.00/34.1k [00:00<?, ?B/s]
100%|███████████████████████████████████████| 34.1k/34.1k [00:00<00:00, 945kB/s]


In [7]:
ds= zf.ZipFile('datasets/titanic.zip')

train_data= pd.read_csv(ds.open('train.csv'))
test_data= pd.read_csv(ds.open('test.csv'))

train_data.shape, test_data.shape

((891, 12), (418, 11))

In [8]:
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [9]:
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [10]:
X_all= pd.concat([train_data[['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']],
                   test_data[['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']]]).set_index('PassengerId')

y_train= train_data[['PassengerId','Survived']].set_index('PassengerId')['Survived']

X_all

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
PassengerId,,,,,,,
1,3,male,22.0,1,0,7.2500,S
2,1,female,38.0,1,0,71.2833,C
3,3,female,26.0,0,0,7.9250,S
4,1,female,35.0,1,0,53.1000,S
5,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...
1305,3,male,NaN,0,0,8.0500,S
1306,1,female,39.0,0,0,108.9000,C
1307,3,male,38.5,0,0,7.2500,S


In [11]:
numeric_columns= ['Age','SibSp','Parch','Fare']
categor_columns= list(filter(lambda x:x not in numeric_columns,X_all.columns))

X_train= X_all.iloc[:len(train_data),:].copy()
X_test= X_all.iloc[len(train_data):].copy()

In [12]:
# in this case we'll only use X_train df. X_test is not labeled

In [13]:
X_train= pre_proc_fillna_num_fts(X_train,numeric_columns,num_type='median')

X_train= pre_proc_fillna_cat_fts(X_train,categor_columns,cat_type='mode')

X_train

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
PassengerId,,,,,,,
1,3,male,22.0,1,0,7.2500,S
2,1,female,38.0,1,0,71.2833,C
3,3,female,26.0,0,0,7.9250,S
4,1,female,35.0,1,0,53.1000,S
5,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...
887,2,male,27.0,0,0,13.0000,S
888,1,female,19.0,0,0,30.0000,S
889,3,female,28.0,1,2,23.4500,S


In [14]:
# one-hot encoding the qualitative features
X_train_ohe= pd.get_dummies(X_train,columns=categor_columns)

X_train_ohe.head()

,Age,SibSp,Parch,Fare,Pclass_1,Pclass_2,Pclass_3,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
PassengerId,,,,,,,,,,,,
1,22.0,1,0,7.2500,0,0,1,0,1,0,0,1
2,38.0,1,0,71.2833,1,0,0,1,0,1,0,0
3,26.0,0,0,7.9250,0,0,1,1,0,0,0,1
4,35.0,1,0,53.1000,1,0,0,1,0,0,0,1
5,35.0,0,0,8.0500,0,0,1,0,1,0,0,1


In [15]:
y_train.head()

PassengerId
1    0
2    1
3    1
4    1
5    0
Name: Survived, dtype: int64

In [16]:
# normalize the numeric columns of dataframe with each value between 0 and 1
X_train= normalize_selected_cols(X_train, numeric_columns)
X_train_ohe= normalize_selected_cols(X_train_ohe, numeric_columns)

# Feature Attribution using Raking - Methods

In [17]:
# return a DataFrame with replace values (mean/median/mode/none to categorical and numeric) to fill train cols. df is post-processed (cat_cols encoded)
def replace_values(df,num_cols,num_type='mean',cat_type='none'):
    cat_values= None
    num_values= None
    
    if (cat_type=='mean'):
        cat_values= df.mean(axis=0).to_frame().T
    elif (cat_type=='median'):
        cat_values= df.median(axis=0).to_frame().T
    elif (cat_type=='mode'):
        cat_values= df.mode(axis=0)
    
    if (num_type=='mode'):
        num_values= df.mode(axis=0)
    elif (num_type=='median'):
        num_values= df.median(axis=0).to_frame().T
    elif (num_type=='mean'):
        num_values= df.mean(axis=0).to_frame().T

    if(cat_type!='none'):
        cat_values[num_cols]= num_values[num_cols]
        return cat_values
    
    return num_values

In [18]:
# train the ML model and return its mean accuracy after n_train runs
def train_model_get_acc_mean(model, x_trn, x_tst, y_trn, y_tst, n_train):
    trainings= []

    for i in range(n_train):

        model.fit(x_trn, y_trn)
        acc= sklearn.metrics.accuracy_score(y_tst, model.predict(x_tst))

        trainings.append(acc)

    return np.mean(trainings)

In [19]:
# re-training is needed because machine learning models typically assume that the train and the test data comes from a similar distribution 
# (Hooker et al., 2018)
# here we return p(x|i) and p(x|ij)
def remove_and_retrain_v1(model, replace_ft_vals, x_trn, x_tst, y_trn, y_tst, n_train):
    acc_no_i= []
    acc_no_ij= []

    n_fts= len(x_trn.columns)

    start= time.time()

    for i in range(n_fts):
        acc_row= []

        # replace the i-th ft with its respective mode/mean to "remove" it. train the ML model and get the mean accuracy
        train_copy_no_i= x_trn.copy()
        train_copy_no_i.loc[:,train_copy_no_i.columns[i]]= replace_ft_vals.iloc[0,i]

        acc_no_i.append(train_model_get_acc_mean(model, train_copy_no_i, x_tst, y_trn, y_tst, n_train))

        for j in range(n_fts):

            if (i!= j):
                # replace the j-th ft with its respective mode/mean to "remove" it. here, we "remove" the i-th and the j-th ft
                # train the ML model and get the mean accuracy
                train_copy_no_ij= train_copy_no_i.copy()
                train_copy_no_ij.loc[:,train_copy_no_ij.columns[j]]= replace_ft_vals.iloc[0,j]

                acc_row.append(train_model_get_acc_mean(model, train_copy_no_ij, x_tst, y_trn, y_tst, n_train))
            else:
                acc_row.append(0)

        acc_no_ij.append(acc_row)

    end= time.time()
    #print("--- %s seconds ---" % np.round((end- start), 2))

    return acc_no_i, acc_no_ij

In [20]:
# here we return | p(x|ij) - p(x|i) |
def get_p_matrix_v1(n_fts, acc_no_i, acc_no_ij):
    p_matrix= np.zeros((n_fts, n_fts))

    for i in range(n_fts):
        pi= acc_no_i[i]
        
        for j in range(n_fts):
            if (i!= j):
                pij= acc_no_ij[i][j]
                
                p_matrix[i][j]= abs(pij- pi)
                
    return p_matrix

In [21]:
# here we return | p(x|ij) - p(x|j) |
def get_p_matrix_v2(n_fts, acc_no_j, acc_no_ij):
    p_matrix= np.zeros((n_fts, n_fts))

    for i in range(n_fts):
        for j in range(n_fts):
            if (i!= j):
                pj= acc_no_j[j]
                pij= acc_no_ij[i][j]
                
                p_matrix[i][j]= abs(pij- pj)
                
    return p_matrix

In [22]:
# here we return (| p(x|ij) - p(x|j) | + | p(x|i) - p(x) |) / 2
def get_p_matrix_v3(n_fts, acc_all, acc_no_i, acc_no_j, acc_no_ij):
    p_matrix= np.zeros((n_fts, n_fts))
    p= acc_all

    for i in range(n_fts):
        pi= acc_no_i[i]
        
        for j in range(n_fts):
            if (i!= j):
                pj= acc_no_j[j]
                pij= acc_no_ij[i][j]
                
                p_matrix[i][j]= (abs(pij- pj)+ abs(pi- p))/ 2
                
    return p_matrix

In [23]:
# here we return (| p(x|ij) - p(x|i) | + | p(x|j) - p(x) |) / 2
def get_p_matrix_v4(n_fts, acc_all, acc_no_i, acc_no_j, acc_no_ij):
    p_matrix= np.zeros((n_fts, n_fts))
    p= acc_all

    for i in range(n_fts):
        pi= acc_no_i[i]
        
        for j in range(n_fts):
            if (i!= j):
                pj= acc_no_j[j]
                pij= acc_no_ij[i][j]
                
                p_matrix[i][j]= (abs(pij- pi)+ abs(pj- p))/ 2
                
    return p_matrix

In [24]:
# the stationary distribution is the fraction of time that the system spends in each state as the number of samples approaches infinity
# it looks like there's not a built-in method to find the stationary distribution

# converts a matrix to a row stochastic matrix - a real square matrix, with each row summing to 1
def to_row_stochastic_matrix(M):
    result= M
    
    for row in result:
        n= sum(row)
        if n> 0:
            row[:]= [f/sum(row) for f in row]
    
    return result

In [25]:
# the stationary distribution - analytical solution
# return 1D array
def stationary_dist_v1(stochastic_matrix):
    
    size_A= stochastic_matrix.shape[1]
    ones= [1]* size_A

    A= np.append(np.transpose(stochastic_matrix)- np.identity(size_A),[ones],axis=0)

    v= np.zeros(size_A+ 1)
    v[size_A]= 1
    v= np.transpose(v)

    stationary= np.linalg.solve(np.transpose(A).dot(A), np.transpose(A).dot(v))

    return stationary

In [26]:
# the stationary distribution - another analytical solution
# return 2D array
def stationary_dist_v2(stochastic_matrix):
    # we have to transpose so that Markov transitions correspond to right multiplying by a column vector
    eigval, eigvec= eigs(stochastic_matrix.T, k=1, which='LM')
    stationary= eigvec/ eigvec.sum()

    # eigs finds complex eigenvalues and eigenvectors, so you'll want the real part.
    stationary= stationary.real

    return stationary

# ML model setup

In [27]:
train, test, labels_train, labels_test= train_test_split(X_train_ohe,y_train,train_size=0.80,random_state=1234)
rf= sklearn.ensemble.RandomForestClassifier(n_estimators=500,n_jobs=2)

#train, test, labels_train, labels_test= train_test_split(X_train_ohe,y_train,train_size=0.80,random_state=1234)
#xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss',use_label_encoder=False)

#train, test, labels_train, labels_test= train_test_split(X_train,y_train,train_size=0.80,random_state=1234)
#cat_fts= [train.columns.to_list().index(col) for col in categor_columns]
#ctb_model= CatBoostClassifier(cat_features=cat_fts,silent=True)

# --- Tests ---

In [28]:
# return a sorted DataFrame with Features and Importances - dataset's original features
def ft_importance_df(importances, ft_names):
    fti= pd.Series(importances, index=ft_names).sort_values(ascending=False).to_frame().reset_index()
    fti= fti.rename(columns= {'index':'Feature',0:'Importance'}, inplace=False)
    fti['Feature'].replace({'Sex_male':'Sex','Sex_female':'Sex',
                           'Pclass_1':'Pclass','Pclass_2':'Pclass','Pclass_3':'Pclass',
                           'Embarked_S':'Embarked','Embarked_Q':'Embarked','Embarked_C':'Embarked'}, inplace=True)
    fti= fti.groupby(['Feature']).sum().sort_values('Importance', ascending=False).reset_index()
    
    return fti

In [29]:
# re-training can result in slightly different models, it is essential to repeat the training process multiple times to ensure that the variance in accuracy is low 
# (Hooker et al., 2018)
repeat_train= 1

num_fts= len(train.columns)

In [30]:
# train the ML model and get the mean accuracy using the entire feature set
acc_all_fts= train_model_get_acc_mean(rf, train, test, labels_train, labels_test, repeat_train)

acc_all_fts

0.8212290502793296

In [31]:
# values to "remove" and retrain
replace_ft= replace_values(train,numeric_columns,num_type='mean',cat_type='median')

replace_ft

,Age,SibSp,Parch,Fare,Pclass_1,Pclass_2,Pclass_3,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,0.362142,0.064782,0.064841,0.064072,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0


In [32]:
# train the ML model and get the mean accuracy removing and retraining columns from the feature set
acc_no_i, acc_no_ij= remove_and_retrain_v1(rf, replace_ft, train, test, labels_train, labels_test, repeat_train)

In [33]:
#print(acc_no_i)
print(np.around(acc_no_i, decimals=3))
print("-----------------------------")
#print(acc_no_ij)
print(np.around(acc_no_ij, decimals=3))

[0.793 0.816 0.81  0.827 0.827 0.821 0.827 0.821 0.821 0.821 0.821 0.816]
-----------------------------
[[0.    0.804 0.832 0.804 0.804 0.799 0.799 0.799 0.81  0.799 0.788 0.804]
 [0.804 0.    0.827 0.799 0.821 0.821 0.821 0.816 0.821 0.816 0.81  0.816]
 [0.832 0.821 0.    0.827 0.827 0.821 0.816 0.816 0.821 0.827 0.804 0.816]
 [0.816 0.804 0.827 0.    0.838 0.827 0.838 0.821 0.827 0.821 0.827 0.821]
 [0.793 0.821 0.821 0.832 0.    0.827 0.827 0.821 0.827 0.821 0.821 0.821]
 [0.804 0.816 0.821 0.832 0.827 0.    0.793 0.821 0.821 0.821 0.821 0.81 ]
 [0.793 0.81  0.821 0.832 0.827 0.799 0.    0.821 0.816 0.821 0.816 0.821]
 [0.799 0.804 0.81  0.832 0.821 0.821 0.821 0.    0.687 0.816 0.821 0.821]
 [0.799 0.81  0.821 0.827 0.827 0.821 0.821 0.693 0.    0.816 0.821 0.816]
 [0.793 0.804 0.832 0.827 0.827 0.821 0.816 0.816 0.821 0.    0.816 0.816]
 [0.788 0.816 0.816 0.832 0.821 0.821 0.821 0.821 0.821 0.821 0.    0.816]
 [0.793 0.816 0.832 0.827 0.827 0.816 0.816 0.816 0.816 0.821 0.816 0. 

In [34]:
# get the probability matrix
p_matrix1= get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

#print(p_matrix1)
print(np.around(p_matrix1, decimals=3))
print("-----------------------------")
#print(p_matrix2)
print(np.around(p_matrix2, decimals=3))
print("-----------------------------")
#print(p_matrix3)
print(np.around(p_matrix3, decimals=3))
print("-----------------------------")
#print(p_matrix4)
print(np.around(p_matrix4, decimals=3))

[[0.    0.011 0.039 0.011 0.011 0.006 0.006 0.006 0.017 0.006 0.006 0.011]
 [0.011 0.    0.011 0.017 0.006 0.006 0.006 0.    0.006 0.    0.006 0.   ]
 [0.022 0.011 0.    0.017 0.017 0.011 0.006 0.006 0.011 0.017 0.006 0.006]
 [0.011 0.022 0.    0.    0.011 0.    0.011 0.006 0.    0.006 0.    0.006]
 [0.034 0.006 0.006 0.006 0.    0.    0.    0.006 0.    0.006 0.006 0.006]
 [0.017 0.006 0.    0.011 0.006 0.    0.028 0.    0.    0.    0.    0.011]
 [0.034 0.017 0.006 0.006 0.    0.028 0.    0.006 0.011 0.006 0.011 0.006]
 [0.022 0.017 0.011 0.011 0.    0.    0.    0.    0.134 0.006 0.    0.   ]
 [0.022 0.011 0.    0.006 0.006 0.    0.    0.128 0.    0.006 0.    0.006]
 [0.028 0.017 0.011 0.006 0.006 0.    0.006 0.006 0.    0.    0.006 0.006]
 [0.034 0.006 0.006 0.011 0.    0.    0.    0.    0.    0.    0.    0.006]
 [0.022 0.    0.017 0.011 0.011 0.    0.    0.    0.    0.006 0.    0.   ]]
-----------------------------
[[0.    0.011 0.022 0.022 0.022 0.022 0.028 0.022 0.011 0.022 0.034 0

In [35]:
# finding the stationary distribution
st_matrix1= np.asarray(p_matrix1)
st_matrix2= np.asarray(p_matrix2)
st_matrix3= np.asarray(p_matrix3)
st_matrix4= np.asarray(p_matrix4)

# now convert to right stochastic matrix - a real square matrix, with each row summing to 1
st_matrix1= to_row_stochastic_matrix(st_matrix1)
st_matrix2= to_row_stochastic_matrix(st_matrix2)
st_matrix3= to_row_stochastic_matrix(st_matrix3)
st_matrix4= to_row_stochastic_matrix(st_matrix4)

print(st_matrix1.sum())
print(st_matrix2.sum())
print(st_matrix3.sum())
print(st_matrix4.sum())

12.0
12.0
12.0
12.0


In [36]:
# get the stationary distribution
stationary_d1= stationary_dist_v1(st_matrix1)
stationary_d2= stationary_dist_v1(st_matrix2)
stationary_d3= stationary_dist_v1(st_matrix3)
stationary_d4= stationary_dist_v1(st_matrix4)

print(stationary_d1)
print("-----------------------------")
print(stationary_d2)
print("-----------------------------")
print(stationary_d3)
print("-----------------------------")
print(stationary_d4)

[0.17923581 0.09711263 0.10608306 0.09380932 0.06948753 0.03615531
 0.05080753 0.11503311 0.1218027  0.04743727 0.03322596 0.04980976]
-----------------------------
[0.11795382 0.05817578 0.12896035 0.06998513 0.04186632 0.06637569
 0.11325819 0.13364187 0.12996466 0.03239888 0.07156397 0.03585534]
-----------------------------
[0.08509514 0.06345766 0.10746437 0.06522453 0.04855544 0.05265888
 0.09840611 0.17056318 0.16538944 0.04397626 0.05612668 0.04308232]
-----------------------------
[0.23809348 0.09266271 0.14445199 0.0928277  0.08192077 0.03013012
 0.06398927 0.05828075 0.06949728 0.0359216  0.02480295 0.06742138]


In [37]:
# feature importance ranking stationary_d1
ft_ranking= pd.Series(stationary_d1, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Age           0.179236
Sex_male      0.121803
Sex_female    0.115033
Parch         0.106083
SibSp         0.097113
Fare          0.093809
Pclass_1      0.069488
Pclass_3      0.050808
Embarked_S    0.049810
Embarked_C    0.047437
Pclass_2      0.036155
Embarked_Q    0.033226
dtype: float64

In [38]:
ft_importance_df(stationary_d1,X_train_ohe.columns)

,Feature,Importance
0,Sex,0.236836
1,Age,0.179236
2,Pclass,0.156450
3,Embarked,0.130473
4,Parch,0.106083
5,SibSp,0.097113
6,Fare,0.093809


In [39]:
# feature importance ranking stationary_d1
ft_ranking= pd.Series(stationary_d2, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Sex_female    0.133642
Sex_male      0.129965
Parch         0.128960
Age           0.117954
Pclass_3      0.113258
Embarked_Q    0.071564
Fare          0.069985
Pclass_2      0.066376
SibSp         0.058176
Pclass_1      0.041866
Embarked_S    0.035855
Embarked_C    0.032399
dtype: float64

In [40]:
ft_importance_df(stationary_d2,X_train_ohe.columns)

,Feature,Importance
0,Sex,0.263607
1,Pclass,0.221500
2,Embarked,0.139818
3,Parch,0.128960
4,Age,0.117954
5,Fare,0.069985
6,SibSp,0.058176


In [41]:
# feature importance ranking stationary_d2
ft_ranking= pd.Series(stationary_d3, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Sex_female    0.170563
Sex_male      0.165389
Parch         0.107464
Pclass_3      0.098406
Age           0.085095
Fare          0.065225
SibSp         0.063458
Embarked_Q    0.056127
Pclass_2      0.052659
Pclass_1      0.048555
Embarked_C    0.043976
Embarked_S    0.043082
dtype: float64

In [42]:
ft_importance_df(stationary_d1,X_train_ohe.columns)

,Feature,Importance
0,Sex,0.236836
1,Age,0.179236
2,Pclass,0.156450
3,Embarked,0.130473
4,Parch,0.106083
5,SibSp,0.097113
6,Fare,0.093809


In [43]:
# feature importance ranking stationary_d2
ft_ranking= pd.Series(stationary_d4, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Age           0.238093
Parch         0.144452
Fare          0.092828
SibSp         0.092663
Pclass_1      0.081921
Sex_male      0.069497
Embarked_S    0.067421
Pclass_3      0.063989
Sex_female    0.058281
Embarked_C    0.035922
Pclass_2      0.030130
Embarked_Q    0.024803
dtype: float64

In [44]:
ft_importance_df(stationary_d1,X_train_ohe.columns)

,Feature,Importance
0,Sex,0.236836
1,Age,0.179236
2,Pclass,0.156450
3,Embarked,0.130473
4,Parch,0.106083
5,SibSp,0.097113
6,Fare,0.093809


In [45]:
eigval, eigvec= eigs(st_matrix3.T, k=1, which='LM')
eigval

array([1.+0.j])

In [46]:
f= open('datasets/FAR_data.txt', 'w')

for i in range(num_fts):
    for j in range(num_fts):
        line= (str(i) + ',' + str(j) + ',' + str(p_matrix1[i][j]) + '\n')
        f.write(line)
        
f.close()

In [47]:
# FROM HERE AND BELOW, ONLY FOR TESTS
#import sklearn.datasets

#iris= sklearn.datasets.load_iris()
#train, test, labels_train, labels_test= train_test_split(iris.data,iris.target,train_size=0.80,random_state=1234)

In [48]:
#rf= sklearn.ensemble.RandomForestClassifier(n_estimators=500,n_jobs=2)
#rf.fit(train, labels_train)
#sklearn.metrics.accuracy_score(labels_test, rf.predict(test))

In [49]:
#xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss',use_label_encoder=False)
#xgb_model.fit(train, labels_train)
#sklearn.metrics.accuracy_score(labels_test, xgb_model.predict(test))

In [50]:
#ctb_model= CatBoostClassifier(silent=True)
#ctb_model.fit(train,labels_train)
#sklearn.metrics.accuracy_score(labels_test,ctb_model.predict(test))